## Classic Environment Preflight

This notebook requires the classic runtime. If this check fails, rebuild with `CLASSIC=1 make notebooks-build`, restart the container, and select kernel **Python 3 (classic-langchain)**.


In [ ]:
import os
import sys

def _classic_fail(reason: str) -> None:
    raise RuntimeError(
        f"Classic runtime preflight failed: {reason}\n"
        "Fix:\n"
        "1. CLASSIC=1 make notebooks-build\n"
        "2. make notebooks-up\n"
        "3. In Jupyter, select kernel: Python 3 (classic-langchain)"
    )

kernel_name = os.getenv("JPY_KERNEL_NAME", "")
prefix = sys.prefix.lower()
if "venv-classic" not in prefix and "classic" not in kernel_name.lower():
    _classic_fail(f"detected sys.prefix={sys.prefix!r}, JPY_KERNEL_NAME={kernel_name!r}")

try:
    import langchain  # noqa: F401
except Exception as exc:
    _classic_fail(f"langchain import failed: {exc}")

print("Classic preflight passed.")


We're almost there:
- We loaded the documents
- We've split them in to relevant parts (splitters)
- We've had the vector database calculate the embeddings and store the related parts

Related Langchain example: https://python.langchain.com/docs/use_cases/question_answering/
> Classic track note: This notebook demonstrates legacy/classic LangChain-era patterns for evaluation and comparison.
> Prefer the modern equivalents in `lessons/2026/langchain/` for current APIs and recommended techniques.


In [ ]:
%pip install -q langchain langchain-openai langchain-community

In [ ]:
%pip install -q python-dotenv
from dotenv import load_dotenv
load_dotenv()

We connect to the previously created vector database

In [ ]:

# Set the embeddings function
from langchain_openai import OpenAIEmbeddings

embeddings_model = OpenAIEmbeddings()

# Create a chromadb client
import chromadb

collection_name="my_langchain"
chroma_client = chromadb.PersistentClient(path="./chromadb")
from langchain_community.vectorstores import Chroma
vectorstore = Chroma(embedding_function=embeddings_model,client=chroma_client,collection_name=collection_name)


Now this is the step we put it all together by using the RetrievalQA chain.
This combines the vectordatabase , the prompt , the llm to return answer.


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)
from _lessonshelper.pretty_print_callback_handler import PrettyPrintCallbackHandler

pretty_print_callback = PrettyPrintCallbackHandler()
llm.callbacks = [pretty_print_callback]

from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm, retriever=vectorstore.as_retriever(), return_source_documents=True
)


For comparison let's first do our query through the regular llm call.
You see it returns 4 authors of the Devops Handbook.

In [ ]:

query = "Who were the people involved in writing the devops Handbook ? If there are multiple authors return them all. Return the result as json and use the field firstname and lastname"


# First via the llm
from langchain.schema import (
    HumanMessage,
)

answer = llm.invoke([HumanMessage(content=query)])
from pprint import pprint
pprint(answer)


Now with the retrieval chain we see it returns 5 authors. The extra one being John Allspaw who wrote the foreword.

In [ ]:

# next via the llm via Retrieval Augmented Generation
answer = qa_chain.invoke({"query": query})
pprint(answer)


This combination of RAG is one of the primary patterns people are using LLMs with their own data.